In [38]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from scipy import stats
import utils
import seaborn as sns
import pingouin as pg
import matplotlib.cm as cm

import statsmodels.stats.power as smp
from statsmodels.stats.anova import AnovaRM
from tqdm import tqdm


from natsort import index_natsorted

# plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['font.family'] = 'Calibri'

path_figs = "./Figs/"

fingers = ['1', '2', '3', '4', '5']

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time


total_sub_num = 20
num_sessions = 1
num_blocks_per_session = 20
num_trials_per_block = 40

# sub_nums = [1,2]
sub_nums = [1,2]

utils.set_figure_style("1col")
sns.color_palette('colorblind')


[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [39]:
# reload utils
import importlib
importlib.reload(utils)

<module 'utils' from '/Users/amin/projects/LearningDynamics/ChordAdaptationDynamics/utils.py'>

In [40]:
subjs_list = utils.read_dat_files_subjs_list(sub_nums)

subjs = pd.concat(subjs_list, ignore_index=True)

subjs['TotalTrialNum'] = (subjs['BN'] - 1) * num_trials_per_block + subjs['TN']
subjs['day'] = subjs['BN'].apply(lambda x: (x - 1) // num_blocks_per_session + 1)
subjs['actualEndForce'] = subjs['endForce'] / subjs['forceGain']
subjs.reset_index(drop=True, inplace=True)



In [41]:
subjs

,BN,TN,subNum,chord,targetVisible,planTime,feedbackTime,iti,forceGain,targetForce,endForce,trialCorr,trialErrorType,TotalTrialNum,day,actualEndForce
0,1,1,1,1,0,2000,2000,1000,1.00,-3.0,-4.913000e+00,1,0,1,1,-4.913000e+00
1,1,2,1,1,0,2000,2000,1000,1.00,-3.0,-4.137000e+00,1,0,2,1,-4.137000e+00
2,1,3,1,1,0,2000,2000,1000,1.00,-3.0,-4.616000e+00,1,0,3,1,-4.616000e+00
3,1,4,1,1,0,2000,2000,1000,1.00,-3.0,-3.528000e+00,1,0,4,1,-3.528000e+00
4,1,5,1,1,0,2000,2000,1000,1.00,-3.0,-3.135000e+00,1,0,5,1,-3.135000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,4,36,2,1,0,2000,2000,1000,0.75,-3.0,-6.277439e+66,0,2,156,1,-8.369918e+66
316,4,37,2,1,0,2000,2000,1000,0.75,-3.0,-3.092000e+00,1,0,157,1,-4.122667e+00
317,4,38,2,1,0,2000,2000,1000,0.75,-3.0,-3.040000e+00,1,0,158,1,-4.053333e+00
318,4,39,2,1,0,2000,2000,1000,0.75,-3.0,-3.120000e+00,1,0,159,1,-4.160000e+00


In [42]:
subjs.columns

Index(['BN', 'TN', 'subNum', 'chord', 'targetVisible', 'planTime',
       'feedbackTime', 'iti', 'forceGain', 'targetForce', 'endForce',
       'trialCorr', 'trialErrorType', 'TotalTrialNum', 'day',
       'actualEndForce'],
      dtype='str')

In [43]:
subjs = subjs[['subNum', 'BN', 'TN', 'TotalTrialNum', 'chord', 'targetVisible', 'forceGain', 'targetForce', 'endForce', 'actualEndForce',
               'trialCorr', 'trialErrorType', 'day']]

In [44]:
# subjs_force = pd.read_csv(utils.path_misc+'subjs_force_train_exp1.csv', sep = '\t')


In [45]:
# # for missing values of endForce, extract from subjs_force
# for i in range(1,6):
#     subjs[f'endForce{i}'] = subjs.apply(lambda row: subjs_force.loc[(subjs_force['subNum'] == row['subNum']) & 
#                                                                     (subjs_force['BN'] == row['BN']) & 
#                                                                     (subjs_force['TN'] == row['TN']), f'endForce{i}'].values[0] 
#                                         if pd.isna(row[f'endForce{i}']) else row[f'endForce{i}'], axis=1)

In [46]:
subjs

,subNum,BN,TN,TotalTrialNum,chord,targetVisible,forceGain,targetForce,endForce,actualEndForce,trialCorr,trialErrorType,day
0,1,1,1,1,1,0,1.00,-3.0,-4.913000e+00,-4.913000e+00,1,0,1
1,1,1,2,2,1,0,1.00,-3.0,-4.137000e+00,-4.137000e+00,1,0,1
2,1,1,3,3,1,0,1.00,-3.0,-4.616000e+00,-4.616000e+00,1,0,1
3,1,1,4,4,1,0,1.00,-3.0,-3.528000e+00,-3.528000e+00,1,0,1
4,1,1,5,5,1,0,1.00,-3.0,-3.135000e+00,-3.135000e+00,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,2,4,36,156,1,0,0.75,-3.0,-6.277439e+66,-8.369918e+66,0,2,1
316,2,4,37,157,1,0,0.75,-3.0,-3.092000e+00,-4.122667e+00,1,0,1
317,2,4,38,158,1,0,0.75,-3.0,-3.040000e+00,-4.053333e+00,1,0,1
318,2,4,39,159,1,0,0.75,-3.0,-3.120000e+00,-4.160000e+00,1,0,1


In [47]:
# aligned_cut_force.to_csv(utils.path_misc+'aligned_cut_force.csv', index = False, sep = '\t')

subjs.to_csv(utils.path_misc+'subjs.csv', index = False, sep = '\t')